# **Integration Word2Vec - Skip-Gram**

## Skip-gram Model

The Skip-gram model is the reverse of CBOW. While CBOW predicts a **center word from its context**, 
Skip-gram does the opposite — it takes a **center word and predicts the surrounding context words**.

The core intuition is simple: words that appear in similar contexts carry similar meanings. 
By training a model to predict context from a word, we force it to learn rich semantic 
relationships between words.

---

### Example

Consider the sentence:

> **The cat sat on the mat**

With a context window of 2, if **"sat"** is our target word:

| Context Word | Target Word |
|---|---|
| The | sat |
| cat | sat |
| on | sat |
| the | sat |

The model asks: *"Given the word 'sat', how likely is each word in the vocabulary to appear nearby?"*

---

### The Windowing Trick

Predicting all context words at once is computationally expensive — the model would need to 
output probabilities across the entire vocabulary **for every context position simultaneously**.

Instead Skip-gram breaks the context into **individual word pairs**, creating one training 
example per context word:

| Target Word | Context Word | Training Pair |
|---|---|---|
| sat | The | (sat → The) |
| sat | cat | (sat → cat) |
| sat | on | (sat → on) |
| sat | the | (sat → the) |

Each pair becomes a **separate prediction task** — the model learns to predict one context 
word at a time from the target. This turns one sentence into many training examples, making 
training more efficient and the embeddings richer.

---

### Skip-gram vs CBOW
```
CBOW:      [The, cat, on, the]  →  "sat"               (many → one)
Skip-gram:       "sat"          →  [The, cat, on, the]  (one → many)
```

| | CBOW | Skip-gram |
|---|---|---|
| Input | context words | center word |
| Output | center word | context words |
| Better for | frequent words | rare words |
| Training speed | faster | slower |
| Embedding quality | good overall | better for small datasets |

---

### Key Takeaway

> Skip-gram treats every context word as a **separate prediction problem** — this simplicity 
> is what makes it powerful and easy to train at scale!

Import Libraries


In [3]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator

In [2]:
corpus = """
The cat sat on the mat and looked around the room.
The dog ran across the yard and jumped over the fence.
The bird flew high above the trees and sang a beautiful song.
The fish swam deep in the ocean and explored the coral reef.
The horse galloped fast across the field and leaped over the gate.

The cat chased the mouse through the narrow alley.
The dog barked loudly at the stranger near the gate.
The bird built a small nest high up in the oak tree.
The fish jumped out of the cold water into the warm air.
The horse trotted slowly along the dusty country road.

A young boy played happily in the green park.
A little girl sang a sweet song under the bright sun.
An old man walked slowly along the quiet river bank.
A tall woman ran quickly through the busy city street.
A small child laughed loudly in the crowded playground.

The sun rose slowly above the distant mountain peaks.
The moon shone brightly over the calm dark ocean.
The stars twinkled softly in the clear night sky.
The wind blew gently through the tall green trees.
The rain fell heavily on the old tin roof.

The teacher explained the difficult lesson to the curious students.
The doctor examined the sick patient in the small clinic.
The chef cooked a delicious meal in the busy restaurant kitchen.
The farmer harvested fresh vegetables from the fertile green field.
The engineer built a strong bridge over the wide river.

The cat curled up and slept near the warm fireplace.
The dog wagged its tail and fetched the round ball.
The bird pecked at the scattered seeds on the ground.
The fish nibbled at the colorful plants in the tank.
The horse neighed loudly and stomped its heavy hooves.

The city streets were crowded with busy rushing people.
The quiet village was surrounded by tall dense forest.
The sandy beach stretched far along the blue coastline.
The rocky mountain path wound steeply up to the peak.
The green meadow was filled with colorful wild flowers.

Books open the mind and expand human knowledge greatly.
Music touches the soul and brings people closer together.
Art expresses deep emotion through color shape and form.
Science explores the unknown and solves complex problems.
Technology connects people across vast distances instantly.

The small kitten played with a soft ball of wool.
The tiny puppy chased its own tail around the garden.
The baby bird chirped loudly waiting for its mother.
The young foal stood shakily on its thin wobbly legs.
The little duckling waddled clumsily toward the pond.

Hard work and dedication always lead to great success.
Kindness and compassion make the world a better place.
Curiosity and creativity drive human progress forward always.
Patience and persistence help overcome the toughest challenges.
Courage and determination turn impossible dreams into reality.
"""

### **Step 1:Tokenization**

#### Vocabulary 

In [4]:
tokienizer = get_tokenizer('basic_english')
tokenized_corpus = tokienizer(corpus)

vocab = build_vocab_from_iterator([tokenized_corpus], specials=["<unk>"])
vocab.set_default_index(vocab["<unk>"])

#### Text Pipeline

In [6]:
text_pipeline = lambda x: vocab(x)
text_pipeline(tokenized_corpus)[0:10] 

[1, 18, 217, 10, 1, 171, 3, 168, 30, 1]

#### Indicies to tokens

In [7]:
indicies_to_tokens = vocab.get_itos()
indicies_to_tokens[217]

'sat'

### **Step 2: Context and Target Tokens**

In [8]:
cbow_data = []
context_size = 2

for i in range (context_size, len(tokenized_corpus)-context_size):
    context = (
        [tokenized_corpus[i-j-1] for j in range(context_size)] +
        [tokenized_corpus[i+j+1] for j in range (context_size)]
    )
    
    target = [tokenized_corpus[i]]
    
    cbow_data.append((context,target))

print(cbow_data[0:10])     
    

[(['cat', 'the', 'on', 'the'], ['sat']), (['sat', 'cat', 'the', 'mat'], ['on']), (['on', 'sat', 'mat', 'and'], ['the']), (['the', 'on', 'and', 'looked'], ['mat']), (['mat', 'the', 'looked', 'around'], ['and']), (['and', 'mat', 'around', 'the'], ['looked']), (['looked', 'and', 'the', 'room'], ['around']), (['around', 'looked', 'room', '.'], ['the']), (['the', 'around', '.', 'the'], ['room']), (['room', 'the', 'the', 'dog'], ['.'])]


### **Step 3: Data Loader**

In [9]:
from torch.utils.data import DataLoader

#### Convert tokens into tensor with indicies

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [11]:
def collate_func (batch):
    target_list, context_list = [],[]
    offsets = [0]
    
    for _context, _target in batch:
        
        process_context = torch.tensor(text_pipeline(_context), dtype=torch.int64)
        context_list.append(process_context)
        
        process_target = vocab[_target]
        target_list.append(process_target)
        
        offsets.append(offsets[-1]+ process_context.size(0))
    
    offsets = torch.tensor(offsets[:-1], dtype=torch.int64)
    target_list = torch.tensor(target_list,dtype=torch.int64)
    context_list = torch.cat(context_list)

    return context_list.to(device), target_list.to(device), offsets.to(device)
        

#### Define Data Loader

In [12]:
batch_size = 32

In [13]:
data_loader = DataLoader(cbow_data,batch_size=batch_size,
                         shuffle=True, collate_fn=collate_func)